# Session Comparison

Load two sessions and compare the same parameter across them — overlay traces, compute deltas, and highlight differences.

**Prerequisites:** Two sessions in the same database with at least one common parameter.

In [ ]:
import sys
sys.path.insert(0, '.')
from sqlrace_helpers import (
    init_sqlrace, load_session, session_summary,
    list_parameters, extract_parameter
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
SESSION_A_GUID = "<REPLACE WITH SESSION A GUID>"
SESSION_B_GUID = "<REPLACE WITH SESSION B GUID>"
COMPARE_PARAM = "Temperature:Sensors"  # Change to a parameter in your sessions

sm = init_sqlrace()

## Load both sessions

In [ ]:
cs_a, sess_a = load_session(sm, SESSION_A_GUID)
print("Session A:")
session_summary(sess_a)

print()
cs_b, sess_b = load_session(sm, SESSION_B_GUID)
print("Session B:")
session_summary(sess_b)

## Extract the comparison parameter

In [ ]:
data_a = extract_parameter(sess_a, COMPARE_PARAM)
data_b = extract_parameter(sess_b, COMPARE_PARAM)

# Convert to relative time (seconds from session start)
time_a = (data_a.index - data_a.index[0]) / 1e9
time_b = (data_b.index - data_b.index[0]) / 1e9

print(f"Session A: {len(data_a)} samples, {time_a.iloc[-1]:.1f} s")
print(f"Session B: {len(data_b)} samples, {time_b.iloc[-1]:.1f} s")

## Overlay trace comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time_a, data_a.values, label="Session A", linewidth=0.8)
ax.plot(time_b, data_b.values, label="Session B", linewidth=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel(COMPARE_PARAM)
ax.set_title(f"Comparison: {COMPARE_PARAM}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Statistical comparison

In [ ]:
stats = pd.DataFrame({
    "Session A": data_a.describe(),
    "Session B": data_b.describe(),
})
stats["Delta"] = stats["Session B"] - stats["Session A"]
display(stats)

## Delta trace

Resample both to a common time grid and plot the difference.

In [ ]:
# Resample to common time grid
common_len = min(len(data_a), len(data_b))
common_time = np.linspace(0, min(time_a.iloc[-1], time_b.iloc[-1]), common_len)

interp_a = np.interp(common_time, time_a, data_a.values)
interp_b = np.interp(common_time, time_b, data_b.values)
delta = interp_b - interp_a

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(common_time, interp_a, label="A", linewidth=0.8)
axes[0].plot(common_time, interp_b, label="B", linewidth=0.8)
axes[0].set_ylabel(COMPARE_PARAM)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(common_time, delta, alpha=0.4, color="red")
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_ylabel("B - A")
axes[1].set_xlabel("Time (s)")
axes[1].grid(True, alpha=0.3)

fig.suptitle(f"Delta: {COMPARE_PARAM}")
plt.tight_layout()
plt.show()

In [ ]:
cs_a.Dispose()
cs_b.Dispose()
print("Sessions closed.")